# JSON Inputs Setup

This notebook loads two JSON files into separate variables:

- `json_edit_data` from `JSON_Edit/ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json`
- `json_whole_model_data` from `JSON Whole Model/ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json`

These will be used for comparison in later cells.

In [10]:
import json
from pathlib import Path

file_name = "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json"

# Resolve workspace root whether current working directory is workspace root or notebook folder
base_dir = Path.cwd()
if not (base_dir / "JSON_Edit").exists() and (base_dir.parent / "JSON_Edit").exists():
    base_dir = base_dir.parent

json_edit_path = base_dir / "JSON_Edit" / file_name
json_whole_model_path = base_dir / "JSON Whole Model" / file_name

with json_edit_path.open("r", encoding="utf-8") as f:
    json_edit_data = json.load(f)

with json_whole_model_path.open("r", encoding="utf-8") as f:
    json_whole_model_data = json.load(f)

print(f"Loaded JSON_Edit file: {json_edit_path}")
print(f"Loaded JSON Whole Model file: {json_whole_model_path}")

Loaded JSON_Edit file: c:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json
Loaded JSON Whole Model file: c:\Git\APS-IFC\JSON Whole Model\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json


In [18]:
import pandas as pd
from pathlib import Path
from IPython.display import HTML, display


def _normalize(value):
    if value is None:
        return ""
    return str(value).strip()


def _build_match_key(item):
    dbid = item.get("DbId")
    external_id = _normalize(item.get("ExternalId"))

    if dbid is not None and external_id:
        return f"DbId:{dbid}|ExternalId:{external_id}"
    if dbid is not None:
        return f"DbId:{dbid}"
    if external_id:
        return f"ExternalId:{external_id}"

    return None


def _find_name_property(item):
    empty = {"category": "", "displayName": "", "value": ""}
    props = item.get("Properties", [])
    if not isinstance(props, list):
        return empty

    for prop in props:
        if not isinstance(prop, dict):
            continue
        if _normalize(prop.get("displayName")).lower() == "name":
            return {
                "category": _normalize(prop.get("category")),
                "displayName": _normalize(prop.get("displayName")),
                "value": _normalize(prop.get("value")),
            }
    return empty


# Build lookup maps by stable object identity
whole_map = {}
for item in json_whole_model_data:
    if isinstance(item, dict):
        key = _build_match_key(item)
        if key:
            whole_map[key] = item

edit_map = {}
for item in json_edit_data:
    if isinstance(item, dict):
        key = _build_match_key(item)
        if key:
            edit_map[key] = item


# Compare renamed objects
rows = []
for key in sorted(set(whole_map.keys()) & set(edit_map.keys())):
    whole_item = whole_map[key]
    edit_item = edit_map[key]

    whole_name = _normalize(whole_item.get("Name"))
    edit_name = _normalize(edit_item.get("Name"))

    if whole_name != edit_name and (whole_name or edit_name):
        whole_name_prop = _find_name_property(whole_item)
        edit_name_prop = _find_name_property(edit_item)

        rows.append(
            {
                "Existing Model Name": whole_name,
                "Edited Name": edit_name,
                "Property Category": whole_name_prop.get("category", "") or edit_name_prop.get("category", ""),
                "Property DisplayName": whole_name_prop.get("displayName", "") or edit_name_prop.get("displayName", ""),
            }
        )

comparison_df = pd.DataFrame(rows)

# Summary + 200-row table (no jump)
total_changed_elements = len(comparison_df)
print(f"Total changed model elements: {total_changed_elements}")

if total_changed_elements == 0:
    print("No renamed objects found between JSON Whole Model and JSON_Edit.")
else:
    max_rows_to_show = 200
    table_to_show = comparison_df.head(max_rows_to_show)

    print(f"Showing {len(table_to_show)} row(s) (limit: {max_rows_to_show}).")

    html_table = table_to_show.to_html(index=False, justify="left", escape=False)
    html_block = f"""
    <div style='text-align:left; font-weight:600; margin:8px 0 6px 0;'>Total changed model elements: {total_changed_elements}</div>
    <div style='text-align:left; font-weight:600; margin:8px 0 6px 0;'>Changed Names Comparison</div>
    <style>
      table.dataframe {{
        margin-left: 0;
        margin-right: auto;
        border-collapse: collapse;
      }}
      table.dataframe th, table.dataframe td {{
        text-align: left !important;
        vertical-align: top;
        padding: 4px 8px;
      }}
    </style>
    {html_table}
    """
    display(HTML(html_block))

# Export to Excel + JSON (used by homepage comparison panel)
workspace_dir = base_dir if (base_dir / "JSON_Edit").exists() else Path.cwd()
export_dir = workspace_dir / "ExportedExcel"
export_dir.mkdir(parents=True, exist_ok=True)

excel_path = export_dir / "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.xlsx"
json_path = export_dir / "ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json"

comparison_df.to_excel(excel_path, index=False)
comparison_df.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"Exported Excel: {excel_path}")
print(f"Exported JSON: {json_path}")

Total changed model elements: 16
Showing 16 row(s) (limit: 200).


Existing Model Name,Edited Name,Property Category,Property DisplayName
1JNL9442575_A-MVS Cable Ladder P1,A-MVS Cable Ladder P1,Item,Name
1JNL9442633_A-Cable Ladder MVS,A-Cable Ladder MVS,Item,Name
1JNL9442631_A-Cable Ladder MVS,A-Cable Ladder MVS,Item,Name
1JNL9441111_A-Cable Ladder 90 450,A-Cable Ladder 90 450,Item,Name
1JNL9441111_A-Cable Ladder 90 450,A-Cable Ladder 90 450,Item,Name
1JNL9442633_A-Cable Ladder MVS,A-Cable Ladder MVS,Item,Name
1JNL9442631_A-Cable Ladder MVS,A-Cable Ladder MVS,Item,Name
1JNL9442631_A-Cable Ladder MVS,A-Cable Ladder MVS,Item,Name
1JNL9441111_A-Cable Ladder 90 450,A-Cable Ladder 90 450,Item,Name
1JNL9442633_A-Cable Ladder MVS,A-Cable Ladder MVS,Item,Name


Exported Excel: c:\Git\APS-IFC\ExportedExcel\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.xlsx
Exported JSON: c:\Git\APS-IFC\ExportedExcel\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json
